In [2]:
import requests
import polars as pl
import os
from datetime import date

# ==============================================================================
# MILESTONE 2.1: RECONCILIATION SPIKE
# Goal: Inspect NESO Constraint Breakdown dataset schema and sample data
# ==============================================================================

# 1. Configuration (No magic numbers)
# Resource ID for 2023-2024 (explicitly chosen because it crosses the Apr 2024 break)
RESOURCE_ID = "24d067d8-1328-452a-9720-21cb691e491e"  
API_URL = "https://api.neso.energy/api/3/action/datastore_search"
SAMPLE_LIMIT = 365  # One row per day for a full year sample
TARGET_BOUNDARIES = ["SCOTEX", "SSEN-S"]
METHODOLOGY_BREAK_DATE = date(2024, 4, 22)

def fetch_sample_data(resource_id: str, limit: int) -> pl.DataFrame:
    """
    Fetches a sample of data from the NESO CKAN Datastore API.
    Fails loudly if the request is unsuccessful or returns no records.
    """
    params = {
        "resource_id": resource_id,
        "limit": limit,
        "offset": 0
    }
    
    print(f"Fetching sample data (limit={limit}) from resource: {resource_id}")
    response = requests.get(API_URL, params=params)
    
    # FAIL LOUDLY: Check for HTTP errors
    response.raise_for_status()
    
    data = response.json()
    
    # FAIL LOUDLY: Check for API-level errors
    if not data.get("success"):
        raise RuntimeError(f"API returned success=false: {data.get('error')}")
    
    records = data["result"]["records"]
    
    # ASSERTION: Protect against empty data silently passing
    assert len(records) > 0, "No records returned from the datastore. Check resource ID."
    
    # Convert to Polars DataFrame
    return pl.DataFrame(records)

def inspect_schema_and_data(df: pl.DataFrame) -> None:
    """
    Answers the 5 critical reconciliation questions from Milestone 2.1.
    """
    print("\n" + "="*80)
    print("RECONCILIATION SPIKE RESULTS")
    print("="*80)
    
    # 1. Schema Inspection
    print("\n1. SCHEMA INSPECTION:")
    print(f"Columns available: {df.columns}")
    
    boundary_keywords = ["boundary", "group", "constraint_type", "region", "zone", "system"]
    boundary_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in boundary_keywords)]
    print(f"Potential boundary/constraint columns: {boundary_cols}")
    
    # 2 & 3. Boundary Breakdown Check
    print("\n2. BOUNDARY BREAKDOWN CHECK:")
    if not boundary_cols:
        print("⚠️ CRITICAL FINDING: No boundary, zone, or group column found in the schema.")
        print("✅ CONCLUSION: This dataset is a SYSTEM-WIDE AGGREGATE.")
        print("🚨 ACTION REQUIRED: Halt pipeline construction for this specific dataset.")
        print("🚨 PIVOT: Initiate Milestone 2.1b (BOALF join) to attribute constraint volumes to SCOTEX (B6) and SSEN-S (B2) boundaries.")
    else:
        boundary_col = boundary_cols[0]
        unique_boundaries = df[boundary_col].unique().drop_nulls().to_list()
        print(f"Unique values in '{boundary_col}': {unique_boundaries}")
        
        has_scotex = any("SCOTEX" in str(b).upper() for b in unique_boundaries)
        has_ssen_s = any("SSEN-S" in str(b).upper() or "SSEN S" in str(b).upper() for b in unique_boundaries)
        
        print(f"  -> Contains SCOTEX? {has_scotex}")
        print(f"  -> Contains SSEN-S? {has_ssen_s}")

    # 4. Methodology Break Check
    print("\n3. METHODOLOGY BREAK CHECK (22 April 2024):")
    date_cols = [col for col in df.columns if "date" in col.lower()]
    if date_cols:
        first_date_col = date_cols[0]
        try:
            df_dates = df.with_columns(pl.col(first_date_col).str.to_date(strict=False))
            min_date = df_dates.select(pl.col(first_date_col).min()).to_series()[0]
            max_date = df_dates.select(pl.col(first_date_col).max()).to_series()[0]
            print(f"  -> Date range in sample: {min_date} to {max_date}")
            
            if min_date <= METHODOLOGY_BREAK_DATE <= max_date:
                print(f"  -> ⚠️ CRITICAL: The sample range SPANS the {METHODOLOGY_BREAK_DATE} methodology break.")
                print("  -> ACTION: Any full-year analysis MUST flag this break. Pre/post April 22 data may not be directly comparable without adjustment.")
        except Exception as e:
            print(f"  -> ⚠️ Could not parse dates: {e}")
    else:
        print("  -> ⚠️ WARNING: No obvious date column found.")

    # 5. Thermal Volume Sanity Check
    print("\n4. THERMAL VOLUME SANITY CHECK:")
    vol_col = "Thermal constraints volume"
    if vol_col in df.columns:
        try:
            df_numeric = df.with_columns(pl.col(vol_col).cast(pl.Float64, strict=False))
            agg = df_numeric.select(pl.col(vol_col).sum().alias("total_thermal_volume_sample"))
            total_volume = agg.to_series()[0]
            print(f"  -> Total 'Thermal constraints volume' in sample (n={len(df)} days): {total_volume:,.0f} MWh")
            print("  -> ACTION: Compare this magnitude against published NESO Constraint Management reports.")
            print("     (Rule of thumb: If sample is ~10k-50k MWh for a few days, annual should be in the low millions of MWh).")
        except Exception as e:
            print(f"  -> ⚠️ Could not aggregate thermal volume: {e}")
    else:
        print(f"  -> ⚠️ WARNING: Column '{vol_col}' not found. Available columns: {df.columns}")

    print("\n" + "="*80)
    print("SPIKE COMPLETE.")
    print("="*80)

if __name__ == "__main__":
    try:
        # Fetch sample data
        sample_df = fetch_sample_data(RESOURCE_ID, limit=SAMPLE_LIMIT)
        
        # Inspect data
        inspect_schema_and_data(sample_df)
        
        # Parquet Handoff Rule: Save to intermediate for audit trail
        os.makedirs("data/intermediate", exist_ok=True)
        output_path = "data/intermediate/01_neso_constraint_breakdown_sample.parquet"
        sample_df.write_parquet(output_path)
        print(f"\n✅ Sample data saved to: {output_path}")
        
    except Exception as e:
        print(f"\n❌ RECONCILIATION SPIKE FAILED LOUDLY: {e}")
        raise

Fetching sample data (limit=365) from resource: 24d067d8-1328-452a-9720-21cb691e491e

RECONCILIATION SPIKE RESULTS

1. SCHEMA INSPECTION:
Columns available: ['_id', 'Date', 'Reducing largest loss cost', 'Increasing system inertia cost', 'Voltage constraints cost', 'Thermal constraints cost', 'Reducing largest loss volume', 'Increasing system inertia volume', 'Voltage constraints volume', 'Thermal constraints volume']
Potential boundary/constraint columns: ['Increasing system inertia cost', 'Increasing system inertia volume']

2. BOUNDARY BREAKDOWN CHECK:
Unique values in 'Increasing system inertia cost': [0, 396, 716, 1221, 1749, 1873, 2391, 2621, 4254, 4876, 5269, 6396, 8130, 9320, 11844, 12957, 13778, 13977, 14727, 14890, 15697, 17091, 18195, 18378, 21858, 22021, 22568, 27355, 35989, 37261, 41391, 42262, 44961, 45229, 48020, 48236, 49907, 50499, 57420, 59152, 60897, 61924, 63262, 63366, 65046, 67985, 71572, 75964, 77521, 78405, 78967, 80704, 90002, 91030, 93687, 94083, 100794, 102228